# SAMS Vision — PPE Detection Training

Train a **YOLOv8** detector for proper wearing of personal protective equipment, used by live SAMS vision.

Maps onto the monitoring HUD:
| SAMS item | Detector classes |
|-----------|------------------|
| `hard_hat` | `helmet` / `no_helmet` |
| `vest` | `vest` (absent ⇒ missing heuristic) |
| `goggles` | `goggles` / `no_goggle` |
| `gloves` | `gloves` / `no_gloves` |

## Datasets

1. **Primary — Ultralytics Construction-PPE**  
   https://docs.ultralytics.com/datasets/detect/construction-ppe/  
   1,416 images, 11 classes (worn + missing gear). Auto-downloads via `construction-ppe.yaml` (~178 MB).

2. **Optional — Mendeley PPE**  
   [Dataset of Personal Protective Equipment (PPE)](https://data.mendeley.com/datasets/zkzghjvpn2)  
   DOI `10.17632/zkzghjvpn2.6` — helmets & reflective vests (YOLO labels). Use as extra fine-tune data if desired.

## Goals
- Download & inspect Construction-PPE (do not train blindly)
- EDA: class counts, sample gallery
- ≥ 4 experiments with different training approaches
- Primary metrics: **mAP50, mAP50-95**, per-class AP for helmet/vest/goggles/gloves
- Export `models/vision/ppe/best.pt` for Django live inference (`ppe_inference.py`)

## Experiment plan
| # | Approach | Model | Key idea |
|---|----------|-------|----------|
| 1 | Scratch-ish baseline | YOLOv8n | Short train, ImageNet backbone |
| 2 | Longer fine-tune | YOLOv8n | More epochs, default aug |
| 3 | Stronger backbone | YOLOv8s | Capacity vs latency |
| 4 | Imbalance-aware | YOLOv8n | Higher box loss / close mosaic off late |
| 5 | Edge export check | best → ONNX (optional) | Deployment sanity |


## 0. Colab setup


In [ ]:
import os
from pathlib import Path

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Running in Colab:", IN_COLAB)
SAMS_ROOT = Path("/content/SAMS") if IN_COLAB else Path("./SAMS_runtime")
for p in [
    SAMS_ROOT / "datasets" / "vision" / "ppe",
    SAMS_ROOT / "models" / "vision" / "ppe",
    SAMS_ROOT / "results" / "vision" / "ppe",
]:
    p.mkdir(parents=True, exist_ok=True)
print("SAMS_ROOT:", SAMS_ROOT.resolve())


In [ ]:
DRIVE_ROOT = None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/SAMS")
else:
    here = Path.cwd().resolve()
    DRIVE_ROOT = here if (here / "manage.py").exists() else (
        here.parent if (here.parent / "manage.py").exists() else Path("./SAMS_drive_mirror")
    )

for p in [
    DRIVE_ROOT / "models" / "vision" / "ppe",
    DRIVE_ROOT / "results" / "vision" / "ppe",
    DRIVE_ROOT / "datasets" / "vision" / "ppe",
]:
    p.mkdir(parents=True, exist_ok=True)

MODELS_DIR = DRIVE_ROOT / "models" / "vision" / "ppe"
RESULTS_DIR = DRIVE_ROOT / "results" / "vision" / "ppe"
DATA_DIR = SAMS_ROOT / "datasets" / "vision" / "ppe"
print("MODELS_DIR:", MODELS_DIR)
print("DATA_DIR:", DATA_DIR)


In [ ]:
import subprocess, sys
pkgs = ["ultralytics>=8.3.0", "seaborn", "tqdm", "pandas", "matplotlib", "pyyaml"]
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
    print("Packages installed.")
else:
    print("Skipping pip install outside Colab (use project venv + ultralytics).")


In [ ]:
import gc, json, shutil, time, warnings
from datetime import datetime, timezone
from typing import Any, Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from ultralytics import YOLO

warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="notebook")

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)


## 1. Dataset — Construction-PPE

Ultralytics downloads the zip on first `YOLO.train(data="construction-ppe.yaml")`.  
We also materialize a local YAML under `DATA_DIR` so paths stay inside SAMS.


In [ ]:
# --- Local YAML pointing at Ultralytics Construction-PPE ----------------------
# Ultralytics resolves `path: construction-ppe` under its datasets root on first train.
# We keep a copy of class names + docs here for EDA after download.

pkg_yaml = Path(__import__("ultralytics").__file__).resolve().parent / "cfg" / "datasets" / "construction-ppe.yaml"
print("Package yaml:", pkg_yaml, "exists=", pkg_yaml.exists())
print(pkg_yaml.read_text() if pkg_yaml.exists() else "missing")

LOCAL_YAML = DATA_DIR / "construction-ppe.yaml"
LOCAL_YAML.write_text(pkg_yaml.read_text())
print("Wrote", LOCAL_YAML)

CLASS_NAMES = {
    0: "helmet", 1: "gloves", 2: "vest", 3: "boots", 4: "goggles",
    5: "none", 6: "Person", 7: "no_helmet", 8: "no_goggle", 9: "no_gloves", 10: "no_boots",
}
SAMS_MAP = {
    "hard_hat": ("helmet", "no_helmet"),
    "vest": ("vest", None),
    "goggles": ("goggles", "no_goggle"),
    "gloves": ("gloves", "no_gloves"),
}


In [ ]:
# --- Trigger dataset download via a 1-epoch dry probe (or reuse cache) --------
# First call downloads ~178 MB to ultralytics datasets/construction-ppe
PROBE = MODELS_DIR / "_probe"
PROBE.mkdir(parents=True, exist_ok=True)

from ultralytics.utils import SETTINGS

print("Ultralytics datasets dir:", SETTINGS.get("datasets_dir"))
datasets_root = Path(SETTINGS.get("datasets_dir", str(Path.home() / "datasets")))
ppe_root = datasets_root / "construction-ppe"

if not (ppe_root / "images" / "train").exists():
    print("Downloading Construction-PPE (first time)…")
    probe_model = YOLO("yolov8n.pt")
    probe_model.train(
        data=str(pkg_yaml),
        epochs=1,
        imgsz=640,
        batch=8,
        device=DEVICE,
        project=str(PROBE),
        name="download_probe",
        exist_ok=True,
        verbose=False,
        workers=2,
    )
    del probe_model
    gc.collect()
else:
    print("Dataset already present at", ppe_root)

assert (ppe_root / "images" / "train").exists(), ppe_root
print("train images:", len(list((ppe_root / "images" / "train").glob("*.*"))))
print("val images:", len(list((ppe_root / "images" / "val").glob("*.*"))))
print("test images:", len(list((ppe_root / "images" / "test").glob("*.*"))))


## 2. EDA — class distribution + sample gallery


In [ ]:
# --- Count YOLO labels -------------------------------------------------------
from collections import Counter

def count_labels(split: str) -> Counter:
    c = Counter()
    label_dir = ppe_root / "labels" / split
    if not label_dir.exists():
        # some releases nest labels beside images
        label_dir = ppe_root / "images" / split
    files = list(Path(str(label_dir).replace("/images/", "/labels/")).glob("*.txt"))
    if not files:
        files = list((ppe_root / "labels" / split).glob("*.txt"))
    for f in files:
        for ln in f.read_text().splitlines():
            if not ln.strip():
                continue
            cid = int(float(ln.split()[0]))
            c[cid] += 1
    return c

split_counts = {}
for split in ("train", "val", "test"):
    split_counts[split] = count_labels(split)
    print(split, dict(split_counts[split]))

rows = []
for split, ctr in split_counts.items():
    for cid, n in ctr.items():
        rows.append({"split": split, "class_id": cid, "class": CLASS_NAMES.get(cid, str(cid)), "count": n})
eda_df = pd.DataFrame(rows)
eda_df.to_csv(RESULTS_DIR / "ppe_class_counts.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 4))
piv = eda_df.pivot_table(index="class", columns="split", values="count", fill_value=0)
piv.plot(kind="bar", ax=ax)
ax.set_title("Construction-PPE — objects per class")
ax.set_ylabel("count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
fig_path = RESULTS_DIR / "ppe_class_distribution.png"
plt.savefig(fig_path, dpi=140, bbox_inches="tight")
plt.show()
print("Saved:", fig_path)


In [ ]:
# --- Sample gallery ----------------------------------------------------------
from PIL import Image

img_dir = ppe_root / "images" / "train"
samples = sorted(img_dir.glob("*.*"))[:8]
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, path in zip(axes.ravel(), samples):
    ax.imshow(Image.open(path).convert("RGB"))
    ax.set_title(path.name[:18], fontsize=8)
    ax.axis("off")
plt.suptitle("Construction-PPE — train samples")
plt.tight_layout()
fig_path = RESULTS_DIR / "ppe_sample_gallery.png"
plt.savefig(fig_path, dpi=120, bbox_inches="tight")
plt.show()
print("Saved:", fig_path)


## 3. Training helpers


In [ ]:
LEDGER: List[Dict[str, Any]] = []


def run_experiment(
    exp_id: str,
    approach: str,
    *,
    weights: str = "yolov8n.pt",
    epochs: int = 30,
    imgsz: int = 640,
    batch: int = 16,
    lr0: float = 0.01,
    close_mosaic: int = 10,
    extra_args: dict | None = None,
) -> dict:
    out_proj = MODELS_DIR / exp_id
    out_proj.mkdir(parents=True, exist_ok=True)
    model = YOLO(weights)
    args = dict(
        data=str(pkg_yaml),
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        device=DEVICE,
        project=str(MODELS_DIR),
        name=exp_id,
        exist_ok=True,
        seed=SEED,
        lr0=lr0,
        close_mosaic=close_mosaic,
        workers=2,
        plots=True,
    )
    if extra_args:
        args.update(extra_args)
    t0 = time.time()
    results = model.train(**args)
    elapsed = time.time() - t0

    # Validate on test split
    best = Path(model.trainer.best) if getattr(model, "trainer", None) else (MODELS_DIR / exp_id / "weights" / "best.pt")
    if not Path(best).exists():
        best = MODELS_DIR / exp_id / "weights" / "best.pt"
    val_model = YOLO(str(best))
    metrics = val_model.val(data=str(pkg_yaml), split="test", imgsz=imgsz, device=DEVICE, plots=False)
    box = metrics.box
    summary = {
        "experiment_id": exp_id,
        "approach": approach,
        "weights_in": weights,
        "epochs": epochs,
        "elapsed_sec": elapsed,
        "map50": float(box.map50),
        "map50_95": float(box.map),
        "best_pt": str(best),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "dataset": "ultralytics/construction-ppe",
    }
    # Per-class AP when available
    try:
        ap = box.maps  # per-class mAP50-95
        names = val_model.names
        summary["ap50_95_per_class"] = {names[i]: float(ap[i]) for i in range(len(ap))}
    except Exception as exc:
        summary["ap_error"] = str(exc)

    with (MODELS_DIR / exp_id / "metadata.json").open("w") as f:
        json.dump(summary, f, indent=2)
    LEDGER.append(summary)
    print(json.dumps({k: summary[k] for k in ("experiment_id", "map50", "map50_95", "elapsed_sec")}, indent=2))

    del model, val_model, results
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return summary


### Experiment 1 — YOLOv8n short baseline


In [ ]:
run_experiment(
    "ppe_exp01_yolov8n_short",
    "Baseline — YOLOv8n, 15 epochs",
    weights="yolov8n.pt",
    epochs=15,
    batch=16 if DEVICE == "cuda" else 8,
)


### Experiment 2 — YOLOv8n longer fine-tune


In [ ]:
run_experiment(
    "ppe_exp02_yolov8n_finetune",
    "Fine-tune — YOLOv8n, 40 epochs",
    weights="yolov8n.pt",
    epochs=40,
    batch=16 if DEVICE == "cuda" else 8,
    lr0=0.01,
)


### Experiment 3 — YOLOv8s (more capacity)


In [ ]:
run_experiment(
    "ppe_exp03_yolov8s",
    "Stronger backbone — YOLOv8s, 30 epochs",
    weights="yolov8s.pt",
    epochs=30,
    batch=8 if DEVICE == "cuda" else 4,
)


### Experiment 4 — YOLOv8n imbalance-aware (late mosaic off)


In [ ]:
run_experiment(
    "ppe_exp04_yolov8n_close_mosaic",
    "Imbalance-aware — YOLOv8n, close_mosaic=5, box=7.5",
    weights="yolov8n.pt",
    epochs=35,
    batch=16 if DEVICE == "cuda" else 8,
    close_mosaic=5,
    extra_args={"box": 7.5, "cls": 0.7},
)


## 4. Compare & promote best weights for live SAMS


In [ ]:
ledger_df = pd.DataFrame(LEDGER).sort_values("map50", ascending=False)
print(ledger_df[["experiment_id", "approach", "map50", "map50_95", "epochs", "elapsed_sec"]].to_string(index=False))
ledger_df.to_csv(RESULTS_DIR / "ppe_experiment_ledger.csv", index=False)

best = ledger_df.iloc[0]
summary = {
    "task": "vision_ppe_detection",
    "dataset": "ultralytics/construction-ppe",
    "optional_mendeley": "10.17632/zkzghjvpn2.6",
    "primary_metric": "map50",
    "best_experiment_id": best["experiment_id"],
    "best_approach": best["approach"],
    "best_metrics": {"map50": float(best["map50"]), "map50_95": float(best["map50_95"])},
    "sams_item_map": SAMS_MAP,
    "models_dir": str(MODELS_DIR),
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}
with (RESULTS_DIR / "ppe_comparison_summary.json").open("w") as f:
    json.dump(summary, f, indent=2)

src = Path(best["best_pt"])
dst = MODELS_DIR / "best.pt"
shutil.copy2(src, dst)
print("Live PPE weights ←", src, "→", dst)

meta = {
    "model_id": "yolov8n_construction_ppe",
    "best_experiment_id": best["experiment_id"],
    "class_names": CLASS_NAMES,
    "sams_map": {
        "hard_hat": ["helmet", "no_helmet"],
        "vest": ["vest"],
        "goggles": ["goggles", "no_goggle"],
        "gloves": ["gloves", "no_gloves"],
    },
    "conf_default": 0.35,
    "dataset": "construction-ppe",
}
with (MODELS_DIR / "metadata.json").open("w") as f:
    json.dump(meta, f, indent=2)
print(json.dumps(summary, indent=2))


### Experiment 5 (optional) — ONNX export


In [ ]:
# Optional edge export of the promoted best.pt
best_pt = MODELS_DIR / "best.pt"
if best_pt.exists():
    m = YOLO(str(best_pt))
    try:
        out = m.export(format="onnx", imgsz=640)
        print("ONNX:", out)
    except Exception as exc:
        print("ONNX export skipped:", exc)
else:
    print("No best.pt yet")


## Next steps (deployment)

1. Ensure `models/vision/ppe/best.pt` is on the SAMS machine (Drive copy or download from Colab).
2. Restart Django / `hw.sh run`.
3. Live phone frames → person detect → **PPE detect** → HUD wearings (overrides mannequin toggles).
4. Disable with `SAMS_VISION_PPE=0` if needed.

Live module: `monitoring/services/ppe_inference.py`
